In [1]:
import joblib
import tensorflow as tf
import pandas as pd

In [2]:
X_historical=joblib.load('X_train.pkl')
y_historical=joblib.load('y_train.pkl')
X_test=joblib.load('X_test.pkl')
y_test=joblib.load('y_test.pkl')
world_cup_processed=pd.read_csv('world_cup_processed.csv')

In [3]:
historical_dates=world_cup_processed.loc[X_historical.index, 'date']
historical_dates=pd.to_datetime(historical_dates)

In [4]:
val_mask=historical_dates.dt.year>=2021
train_mask=historical_dates.dt.year<2021

In [5]:
X_train, y_train = X_historical[train_mask], y_historical[train_mask]
X_val, y_val = X_historical[val_mask], y_historical[val_mask]

In [6]:

print(" Remaining missing numbers in X_test:", X_test.isna().sum().sum())

 Remaining missing numbers in X_test: 0


In [7]:
train_dataset=tf.data.Dataset.from_tensor_slices((X_train.values, y_train.values)).shuffle(len(X_train)).batch(32).prefetch(1)
val_dataset=tf.data.Dataset.from_tensor_slices((X_val.values, y_val.values)).batch(32).prefetch(1)
test_dataset=tf.data.Dataset.from_tensor_slices((X_test.values, y_test.values)).batch(32).prefetch(1)

print(f"Train matches: {len(X_train)} | Val matches: {len(X_val)} | Test (World Cup) matches: {len(X_test)}")

Train matches: 21197 | Val matches: 5584 | Test (World Cup) matches: 72


In [8]:
norm_layer=tf.keras.layers.Normalization()
norm_layer.adapt(X_train.values)

In [9]:
model=tf.keras.Sequential([
    norm_layer,
    tf.keras.layers.Dense(64, activation='elu', kernel_initializer='he_normal'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='elu', kernel_initializer='he_normal'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(3, activation='softmax')
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train_dataset, validation_data=val_dataset, epochs=30)

Epoch 1/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5067 - loss: 1.0368 - val_accuracy: 0.5734 - val_loss: 0.9352
Epoch 2/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5364 - loss: 0.9799 - val_accuracy: 0.5750 - val_loss: 0.9334
Epoch 3/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5425 - loss: 0.9705 - val_accuracy: 0.5761 - val_loss: 0.9392
Epoch 4/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5419 - loss: 0.9685 - val_accuracy: 0.5756 - val_loss: 0.9299
Epoch 5/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5449 - loss: 0.9667 - val_accuracy: 0.5770 - val_loss: 0.9321
Epoch 6/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5445 - loss: 0.9649 - val_accuracy: 0.5734 - val_loss: 0.9351
Epoch 7/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5453 - loss: 0.9620 - val_accuracy: 0.5740 - val_loss: 0.9331
Epoch 8/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5469 - loss: 0.9633 - val_accuracy: 0.

In [10]:
# Making the network wide and deep

In [11]:
model = tf.keras.Sequential([
    norm_layer,
    tf.keras.layers.Dense(128, activation='elu', kernel_initializer='he_normal'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(64, activation='elu', kernel_initializer='he_normal'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(32, activation='elu', kernel_initializer='he_normal'),
    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train_dataset, validation_data=val_dataset, epochs=30)

Epoch 1/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5181 - loss: 1.0114 - val_accuracy: 0.5723 - val_loss: 0.9318
Epoch 2/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5409 - loss: 0.9728 - val_accuracy: 0.5779 - val_loss: 0.9329
Epoch 3/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5449 - loss: 0.9679 - val_accuracy: 0.5738 - val_loss: 0.9368
Epoch 4/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5453 - loss: 0.9670 - val_accuracy: 0.5734 - val_loss: 0.9327
Epoch 5/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5442 - loss: 0.9656 - val_accuracy: 0.5723 - val_loss: 0.9318
Epoch 6/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5455 - loss: 0.9630 - val_accuracy: 0.5772 - val_loss: 0.9291
Epoch 7/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5498 - loss: 0.9616 - val_accuracy: 0.5765 - val_loss: 0.9275
Epoch 8/30
663/663 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5463 - loss: 0.9633 - val_accuracy: 0.

In [12]:
nn_probabilities = model.predict(test_dataset)
test_mask = world_cup_processed['date'] >= '2026-06-11'

simulation_results=world_cup_processed[test_mask].copy()

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step


In [13]:
print(X_test.isna().sum())
print("\n--- Rows with missing data ---")
print(X_test[X_test.isna().any(axis=1)])

home_form_scored       0
home_form_conceded     0
home_class_scored      0
home_class_conceded    0
away_form_scored       0
away_form_conceded     0
away_class_scored      0
away_class_conceded    0
dtype: int64

--- Rows with missing data ---
Empty DataFrame
Columns: [home_form_scored, home_form_conceded, home_class_scored, home_class_conceded, away_form_scored, away_form_conceded, away_class_scored, away_class_conceded]
Index: []


In [14]:
nn_probabilities

array([[0.2810241 , 0.2876226 , 0.4313533 ],
       [0.25548014, 0.25213513, 0.49238482],
       [0.17646366, 0.20917639, 0.6143599 ],
       [0.25787264, 0.24604386, 0.49608344],
       [0.38046473, 0.30086598, 0.31866926],
       [0.49503818, 0.24619083, 0.25877094],
       [0.28590354, 0.27028346, 0.443813  ],
       [0.22047918, 0.19360888, 0.58591187],
       [0.19039148, 0.16115211, 0.6484565 ],
       [0.19712973, 0.22653526, 0.57633495],
       [0.39807546, 0.22505186, 0.37687266],
       [0.3459067 , 0.24104382, 0.41304946],
       [0.20275493, 0.18653107, 0.6107141 ],
       [0.30208063, 0.28217617, 0.41574317],
       [0.24851377, 0.22058466, 0.5309015 ],
       [0.16699661, 0.16431272, 0.6686906 ],
       [0.2746015 , 0.2008366 , 0.5245619 ],
       [0.3745865 , 0.29354727, 0.33186617],
       [0.22000553, 0.21444991, 0.5655446 ],
       [0.15397565, 0.17668374, 0.66934055],
       [0.26224586, 0.23600852, 0.50174564],
       [0.29449728, 0.284993  , 0.4205097 ],
       [0.

In [15]:
simulation_results['nn_prob_away_win'] = nn_probabilities[:, 0]
simulation_results['nn_prob_draw'] = nn_probabilities[:, 1]
simulation_results['nn_prob_home_win'] = nn_probabilities[:, 2]

In [16]:
import joblib

xgb_model = joblib.load("baseline_xgb_model.pkl")
xgb_probabilities = xgb_model.predict_proba(X_test)


simulation_results['xgb_prob_home_win'] = xgb_probabilities[:, 2]
simulation_results['xgb_prob_draw'] = xgb_probabilities[:, 1]
simulation_results['xgb_prob_away_win'] = xgb_probabilities[:, 0]

print(" --- ORACLE COMPARISON: NEURAL NETWORK vs. XGBOOST --- \n")
for i in range(5):
    home = simulation_results.iloc[i]['home_team']
    away = simulation_results.iloc[i]['away_team']

    print(f" {home} vs {away}")
    print(f"    NN  -> Home Win: {simulation_results.iloc[i]['nn_prob_home_win']:.1%}, Draw: {simulation_results.iloc[i]['nn_prob_draw']:.1%}, Away Win: {simulation_results.iloc[i]['nn_prob_away_win']:.1%}")
    print(f"    XGB -> Home Win: {simulation_results.iloc[i]['xgb_prob_home_win']:.1%}, Draw: {simulation_results.iloc[i]['xgb_prob_draw']:.1%}, Away Win: {simulation_results.iloc[i]['xgb_prob_away_win']:.1%}")
    print("-" * 70)

 --- ORACLE COMPARISON: NEURAL NETWORK vs. XGBOOST --- 

 Mexico vs South Africa
    NN  -> Home Win: 43.1%, Draw: 28.8%, Away Win: 28.1%
    XGB -> Home Win: 54.2%, Draw: 26.6%, Away Win: 19.3%
----------------------------------------------------------------------
 South Korea vs Czech Republic
    NN  -> Home Win: 49.2%, Draw: 25.2%, Away Win: 25.5%
    XGB -> Home Win: 52.9%, Draw: 22.6%, Away Win: 24.4%
----------------------------------------------------------------------
 Canada vs Bosnia and Herzegovina
    NN  -> Home Win: 61.4%, Draw: 20.9%, Away Win: 17.6%
    XGB -> Home Win: 66.6%, Draw: 20.2%, Away Win: 13.2%
----------------------------------------------------------------------
 United States vs Paraguay
    NN  -> Home Win: 49.6%, Draw: 24.6%, Away Win: 25.8%
    XGB -> Home Win: 36.6%, Draw: 34.0%, Away Win: 29.4%
----------------------------------------------------------------------
 Brazil vs Morocco
    NN  -> Home Win: 31.9%, Draw: 30.1%, Away Win: 38.0%
    XGB -> 

In [17]:
import pandas as pd
import numpy as np

tournament_df = world_cup_processed[test_mask].copy()

tournament_df['nn_prob_away_win'] = nn_probabilities[:, 0]
tournament_df['nn_prob_draw']     = nn_probabilities[:, 1]
tournament_df['nn_prob_home_win'] = nn_probabilities[:, 2]

xgb_probs = xgb_model.predict_proba(X_test)
tournament_df['xgb_prob_away_win'] = xgb_probs[:, 0]
tournament_df['xgb_prob_draw']     = xgb_probs[:, 1]
tournament_df['xgb_prob_home_win'] = xgb_probs[:, 2]

display_columns = [
    'date', 'home_team', 'away_team',
    'xgb_prob_home_win', 'nn_prob_home_win',
    'xgb_prob_draw', 'nn_prob_draw',
    'xgb_prob_away_win', 'nn_prob_away_win'
]

oracle_simulation_matrix = tournament_df[display_columns]

oracle_simulation_matrix.to_csv("world_cup_oracle_predictions.csv", index=False)
print("Oracle Simulation Matrix successfully generated and saved to 'world_cup_oracle_predictions.csv'!")

Oracle Simulation Matrix successfully generated and saved to 'world_cup_oracle_predictions.csv'!


In [18]:
# 1. Isolate the 72 World Cup matches with group identifiers
# (Assuming your original dataset has 'group' or we keep the basic match info)
tournament_df = world_cup_processed[test_mask].copy()

# 2. Structure the exact columns Streamlit will need to compute standings
dashboard_export = tournament_df[[
    'date', 'home_team', 'away_team'
]].copy()

# Add the probabilities we worked so hard to clean and generate
dashboard_export['xgb_home_win'] = xgb_probabilities[:, 2]
dashboard_export['xgb_draw']     = xgb_probabilities[:, 1]
dashboard_export['xgb_away_win'] = xgb_probabilities[:, 0]

dashboard_export['nn_home_win']  = nn_probabilities[:, 2]
dashboard_export['nn_draw']      = nn_probabilities[:, 1]
dashboard_export['nn_away_win']  = nn_probabilities[:, 0]

# 3. Save this as your master dashboard engine asset
dashboard_export.to_csv("dashboard_group_matches.csv", index=False)
print("📦 Master Simulation Engine Asset saved to 'dashboard_group_matches.csv'!")

📦 Master Simulation Engine Asset saved to 'dashboard_group_matches.csv'!


In [19]:
# Official 48-Team FIFA World Cup 2026 Group Mappings
group_mappings = {
    # Group A
    "Mexico": "Group A", "South Africa": "Group A", "South Korea": "Group A", "Czech Republic": "Group A", "Czechia": "Group A",
    # Group B
    "Canada": "Group B", "Bosnia and Herzegovina": "Group B", "Qatar": "Group B", "Switzerland": "Group B",
    # Group C
    "Brazil": "Group C", "Morocco": "Group C", "Haiti": "Group C", "Scotland": "Group C",
    # Group D
    "United States": "Group D", "Paraguay": "Group D", "Australia": "Group D", "Turkey": "Group D", "Türkiye": "Group D",
    # Group E
    "Germany": "Group E", "Curaçao": "Group E", "Ivory Coast": "Group E", "Côte d'Ivoire": "Group E", "Ecuador": "Group E",
    # Group F
    "Netherlands": "Group F", "Japan": "Group F", "Sweden": "Group F", "Tunisia": "Group F",
    # Group G
    "Belgium": "Group G", "Egypt": "Group G", "Iran": "Group G", "IR Iran": "Group G", "New Zealand": "Group G",
    # Group H
    "Spain": "Group H", "Cape Verde": "Group H", "Cabo Verde": "Group H", "Saudi Arabia": "Group H", "Uruguay": "Group H",
    # Group I
    "France": "Group I", "Senegal": "Group I", "Iraq": "Group I", "Norway": "Group I",
    # Group J
    "Argentina": "Group J", "Algeria": "Group J", "Austria": "Group J", "Jordan": "Group J",
    # Group K
    "Portugal": "Group K", "DR Congo": "Group K", "Congo DR": "Group K", "Uzbekistan": "Group K", "Colombia": "Group K",
    # Group L
    "England": "Group L", "Croatia": "Group L", "Ghana": "Group L", "Panama": "Group L"
}

# Attach the group mappings back to your dashboard dataframe file for context
simulation_results['home_group'] = simulation_results['home_team'].map(group_mappings)
simulation_results['away_group'] = simulation_results['away_team'].map(group_mappings)

# Overwrite your dashboard export file with the clean group tags included
simulation_results.to_csv("dashboard_group_matches.csv", index=False)
print("Groups successfully integrated! Saved to 'dashboard_group_matches.csv'.")

Groups successfully integrated! Saved to 'dashboard_group_matches.csv'.
